# Standard-Error Sensitivity: One-Way Entity Clustering vs. Driscoll-Kraay

Most papers in the globalisation-vs-welfare-state literature report one-way
standard errors clustered at the country level. That is consistent under
within-country serial correlation *and* cross-sectional independence of the
errors.

The second assumption is the fragile one in a macro panel with 32 OECD
countries exposed to a handful of common shocks (oil shocks, the 2008 GFC,
COVID-19). We ran the Pesaran (2004) CD test on the residuals of each
baseline regression (see `outputs/tables/residual_cd_test.tex`) and **reject
the null of cross-sectional independence for every globalisation index**.
Under that kind of strong cross-sectional dependence (CSD) one-way entity
clustering is known to under-estimate standard errors.

This notebook therefore refits the baseline spec three ways and puts the
standard errors side by side so reviewers can see the sensitivity directly:

1. **One-way entity clustering** — the literature convention.
2. **Two-way clustering (entity + time)** — the project default.
3. **Driscoll-Kraay kernel SEs** — nonparametric, robust to serial correlation
   *and* arbitrary cross-sectional dependence.

All three cluster choices use the **same point estimates**; only the standard
errors (and therefore the p-values / stars) differ.

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

from analysis.robustness import export_se_comparison_table
from clean.panel_utils import add_welfare_regimes
from clean.utils import load_config

config = load_config()
master = add_welfare_regimes(
    pd.read_parquet(REPO_ROOT / "data" / "final" / "master_dataset.parquet")
)
print(f"Master panel: {master.shape[0]} rows, {master['iso3'].nunique()} countries")

## Run the comparison

`export_se_comparison_table` refits the baseline
`sstran ~ idx_{t-1} + controls_{t-1}` PanelOLS (entity + time FEs) three
times per globalisation index — once per covariance estimator — and writes a
LaTeX table. We inline the DataFrame here so the numbers are visible in the
notebook.

In [ ]:
tables_dir = REPO_ROOT / "outputs" / "tables"
out_path = export_se_comparison_table(master, config, out_dir=tables_dir)
print(f"LaTeX table: {out_path}")

In [ ]:
# Reproduce the underlying numbers as a DataFrame for inline inspection
from analysis.regression_utils import prepare_regression_data, run_panel_ols, significance_stars
from clean.panel_utils import create_lags

indices = config.get("indices", ["KOFGI", "KOFEcGI", "KOFSoGI", "KOFPoGI"])
ctrls = config.get("controls", ["ln_gdppc", "inflation_cpi", "deficit", "debt", "ln_population", "dependency_ratio"])
dep_var = config.get("dependent_var", "sstran")

rows = []
for idx_name in indices:
    reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(reg_data, dep_var, indep, lagged_ctrls, interactions=False)

    one_way = run_panel_ols(ols_data, dep_var, exog, cluster_entity=True, cluster_time=False)
    two_way = run_panel_ols(ols_data, dep_var, exog, cluster_entity=True, cluster_time=True)
    dk = run_panel_ols(ols_data, dep_var, exog, cov_type="kernel")

    for label, res in [("One-way entity", one_way), ("Two-way", two_way), ("Driscoll-Kraay", dk)]:
        rows.append({
            "Index": idx_name,
            "SE type": label,
            "Coef": round(float(res.params[indep]), 4),
            "SE": round(float(res.std_errors[indep]), 4),
            "t-stat": round(float(res.tstats[indep]), 2),
            "p-value": round(float(res.pvalues[indep]), 4),
            "Stars": significance_stars(res.pvalues[indep]),
        })

comp = pd.DataFrame(rows)
comp

## Reading the table

- The **coefficient column is identical** across the three SE variants — we are
  only changing the variance estimator, not the point estimate.
- If the one-way entity SE is the smallest and the Driscoll-Kraay SE the
  largest, that is the textbook footprint of cross-sectional dependence
  pulling the standard errors up once the common-shock variance is
  acknowledged.
- A significance result that survives all three columns is robust. One that
  only holds under one-way entity clustering should be reported with caveats,
  because the Pesaran CD test tells us that estimator's independence
  assumption is violated in this panel.

## Why we report both

Reviewers who know the literature expect one-way entity clustering — so it
belongs in the main table. But the CD-test evidence means we cannot stop
there. Driscoll-Kraay is the most conservative of the three and is the
appropriate default when the residuals exhibit strong CSD. Reporting both
signals that we are aware of the tension and have not cherry-picked the
estimator that delivers the tightest p-values.